In [1]:
import numpy as np
import json
import chipsplitting as cs
from chipsplitting import pairing_matrix, PascalForm, LinearForm
from chipsplitting.hyperfield import HyperfieldVector as HV, HyperfieldHomogeneousLinearSystem as HLinSystem, grid_iter, HyperfieldLinearForm

In [2]:
def countValidConfigsForContractions(pos_support_size, contraction_size, degree="even", use_extra_constraints = False):
    assert degree == "even" or degree == "odd", "degree must be 'even' or 'odd'"

    d = contraction_size * 3 - 1

    if d % 2 == 0 and degree == "odd":
        d += 1
    elif d % 2 == 1 and degree == "even":
        d += 1
        
    base_types = ["diag", "row", "col"]
    A = [PascalForm(d, b, k) for b in base_types for k in range(contraction_size)] + [PascalForm(d, b, k) for b in base_types for k in range(d - contraction_size + 1, d + 1)]
    
    if use_extra_constraints:
        # (1, d-1) see apple notes 3242444442
        A = A + [PascalForm(d, 'diag', i) - PascalForm(d, 'diag', j) for i,j in [(0,1), (0,2), (0,3), (0,4), (0,d-1), (0,d-2), (0,d-3), (0,d-4), (1,2), (1,3), (1, d), (1,d-1), (1,d-4), (1,d-2), (1,d-3), (2,d), (2,d-1), (2,d-3), (2, d-4), (3,d), (3,d-1), (3,d-2), (3, d-4)]]
        A = A + [PascalForm(d, 'diag', i) - PascalForm(d, 'diag', j) for i,j in [(d-4,d), (d-3,d), (d-2,d), (d-2,d-1), (d-1,d-2), (d-1,d-3), (d-1,d)]]
       
    A = [p.to_hyperfield().contract(contraction_size) for p in A if p != LinearForm.zero(d)]  
    linear_system = HLinSystem(A)
    solutions = linear_system.quick_solve_loop(pos_support_size)
    
    return solutions

In [3]:
%%time
n = 6
contraction_size = 5
res1 = countValidConfigsForContractions(n, contraction_size, "even", use_extra_constraints = True)
print(f"Number of configurations for even d: {len(res1)}")

res2 = countValidConfigsForContractions(n, contraction_size, "odd", use_extra_constraints = True)
print(f"Number of configurations for odd d: {len(res2)}")

Number of configurations for even d: 106806
Number of configurations for odd d: 110272
CPU times: user 2.16 s, sys: 18.5 ms, total: 2.18 s
Wall time: 2.18 s


## Filter

We want contractable Pascal forms that act as a filter.

In [58]:
def absolute(d, c):
    if c == 'd-1':
        return d-1
    if c == 'd-2':
        return d-2
    if c == 'd-3':
        return d-3
    if c == 'd-4':
        return d-4
    if c == 'd-0':
        return d
    return c

def rel(d, contraction_size, index):
    assert index < contraction_size or index > d - contraction_size
    return f"d-{d - index}" if index > contraction_size else index
    
def sign(x):
    return np.sign(x)

def has_inc_c(p, col, contraction_size):
    for row in range(contraction_size, p.degree - contraction_size - col):
        if p.support_pos[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_pos[cs.utils.get_array_index(col, row)] < p.support_pos[cs.utils.get_array_index(col, row + 1)]:
                return False
        if p.support_neg[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_neg[cs.utils.get_array_index(col, row)] > p.support_neg[cs.utils.get_array_index(col, row + 1)]:
                return False
    return True

def has_dec_c(p, col, contraction_size):
    for row in range(contraction_size, p.degree - contraction_size - col):
        if p.support_pos[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_pos[cs.utils.get_array_index(col, row)] > p.support_pos[cs.utils.get_array_index(col, row + 1)]:
                return False
        if p.support_neg[cs.utils.get_array_index(col, row)] > 0:
            if not p.support_neg[cs.utils.get_array_index(col, row)] < p.support_neg[cs.utils.get_array_index(col, row + 1)]:
                return False
    return True

def is_constant(p, col, contraction_size):
    for row in range(contraction_size, p.degree - contraction_size - col):
        if p.support_pos[cs.utils.get_array_index(col, row)] != p.support_pos[cs.utils.get_array_index(col, contraction_size)]:
            return False

        if p.support_neg[cs.utils.get_array_index(col, row)] != p.support_neg[cs.utils.get_array_index(col, contraction_size)]:
            return False
    return True

def is_contractable(p, contraction_size = 5):
    assert p.degree >= contraction_size * 3 - 1

    # check b
    for row in range(contraction_size):
        for col in range(contraction_size, p.degree - contraction_size - row + 1):
            if sign(p.support_pos[cs.utils.get_array_index(col, row)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size, row)]):
                return False

            if sign(p.support_neg[cs.utils.get_array_index(col, row)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size, row)]):
                return False

    # check c
    for col in range(contraction_size):
        for row in range(contraction_size, p.degree - contraction_size - col + 1):
            if sign(p.support_pos[cs.utils.get_array_index(col, row)]) != sign(p.support_pos[cs.utils.get_array_index(col, contraction_size)]):
                return False

            if sign(p.support_neg[cs.utils.get_array_index(col, row)]) != sign(p.support_neg[cs.utils.get_array_index(col, contraction_size)]):
                return False

    # check d1
    for j in range(contraction_size):
        for i in range(contraction_size, p.degree - j - contraction_size + 1, 2):
            if sign(p.support_pos[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size, p.degree - j - contraction_size)]):
                return False
                
            if sign(p.support_neg[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size, p.degree - j - contraction_size)]):
                return False

    # check d2
    for j in range(contraction_size):
        for i in range(contraction_size + 1, p.degree - j - contraction_size + 1, 2):
            if sign(p.support_pos[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_pos[cs.utils.get_array_index(contraction_size + 1, p.degree - j - contraction_size - 1)]):
                return False
                
            if sign(p.support_neg[cs.utils.get_array_index(i, p.degree - i - j)]) != sign(p.support_neg[cs.utils.get_array_index(contraction_size + 1, p.degree - j - contraction_size - 1)]):
                return False

    return True


"""
    Given an expression of pascal forms, find all units such that the expression is contractable.
"""
def find_contractables(lin_combinations, ops, contraction_size):
    import itertools
    
    d_begin = contraction_size * 3 - 1 + 3
    d_end = contraction_size * 3 - 1 + 6
        
    res = []
    combinations = {}

    for d in range(d_begin, d_end + 1):
        units = list(range(contraction_size)) + list(range(d - contraction_size + 1, d + 1))
        bases = [[(PascalForm(d, b, k), rel(d, contraction_size, k)) for k in units] for b in lin_combinations]

        for comb in itertools.product(*bases):
            form = LinearForm.zero(d)
            units = []
            for i, t in enumerate(comb):
                p, u = t
                units.append(u)
                if i == 0 or ops[i-1] == "+":
                    form += p
                else:
                    form -= p
            units = tuple(units)
            if units not in combinations:
                combinations[units] = {}
            combinations[units][d] = {"contractable": is_contractable(form), "form": form.to_hyperfield().contract(contraction_size)}
                
    for units, dict in combinations.items():
        is_valid = True
        for d in dict.keys():
            if d % 2 == 0:
                if dict[d]["contractable"] == False or dict[d]["form"] != dict[d_begin if d_begin % 2 == 0 else d_begin + 1]["form"]:
                    is_valid = False
                    break 
            else:
                if dict[d]["contractable"] == False or dict[d]["form"] != dict[d_begin if d_begin % 2 == 1 else d_begin + 1]["form"]:
                    is_valid = False
                    break
                
        if is_valid:
            res.append(units)

    return res

def hyperfield_vector_from_support(d, support_pos, support_neg):
    w = [-1] + [0] * (d-1)
    for x in support_pos:
        w[x] = 1
    return HV(w)
    
def is_root(hyperfield_forms, w):
    for p in hyperfield_forms:
        y = p(w)
        if not np.isnan(y) and y != 0:
            return False
    return True



def invert(p):
    return HyperfieldLinearForm(p.support_neg, p.support_pos)

### Generate some new contractable pascal forms

In [6]:
3678, 3034, 3716

(3678, 3034, 3716)

In [60]:
d = 16
for comb, ops in [(("row", "row"), ("-"))]:
    for units in find_contractables(comb, ops, 5):
        p = PascalForm(16, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, 5)] > 0 or p.support_neg[cs.utils.get_array_index(col, 5)] > 0:
                    if not has_inc_c(p, col, 5) and not has_dec_c(p,col,5) and not is_constant(p,col,5):
                        print(p)

In [70]:
d = 16
for comb, ops in [(("diag", "row"), ("+"))]:
    for units in find_contractables(comb, ops, 5):
        p = PascalForm(16, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, 5)] > 0 or p.support_neg[cs.utils.get_array_index(col, 5)] > 0:
                    if not has_inc_c(p, col, 5) and not has_dec_c(p,col,5) and not is_constant(p,col,5):
                        print(p)

In [73]:
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1

if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
combinations = [
    (("row", "row"), ("-")),
    (("row", "col"), ("-")),
    (("row", "diag"), ("-")),
    (("col", "col"), ("-")),
    (("col", "diag"), ("-")),
    (("diag", "diag"), ("-")),
    #(("diag", "diag", "col", "row"), ("-", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    #(("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if not has_inc_c(p, col, contraction_size) and not has_dec_c(p,col,contraction_size) and not is_constant(p,col,contraction_size):
                        print(p)

In [75]:
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1

if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
combinations = [
    #(("diag", "diag", "col", "row"), ("-", "+", "+")),
    (("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    #(("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    #(("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if not has_inc_c(p, col, contraction_size) and not has_dec_c(p,col,contraction_size) and not is_constant(p,col,contraction_size):
                        print(p)

In [76]:
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1

if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
combinations = [
    (("diag", "diag", "diag", "row", "col"), ("-", "+", "+", "+")),
    (("diag", "diag", "diag", "row", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if not has_inc_c(p, col, contraction_size) and not has_dec_c(p,col,contraction_size) and not is_constant(p,col,contraction_size):
                        print(p)

In [77]:
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1

if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
combinations = [
    (("row", "row", "col", "diag"), ("+", "+", "+", "+")),
    (("row", "row", "col", "diag"), ("+", "+", "+", "-")),
    (("row", "row", "col", "diag"), ("+", "+", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            for col in range(contraction_size):
                if p.support_pos[cs.utils.get_array_index(col, contraction_size)] > 0 or p.support_neg[cs.utils.get_array_index(col, contraction_size)] > 0:
                    if not has_inc_c(p, col, contraction_size) and not has_dec_c(p,col,contraction_size) and not is_constant(p,col,contraction_size):
                        print(p)

#### Build Filter For Even Contraction Forms

In [29]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
FILTER = set()

combinations = [
    (("row", "row"), ("-")),
    (("row", "col"), ("-")),
    (("row", "diag"), ("-")),
    (("col", "col"), ("-")),
    (("col", "diag"), ("-")),
    (("diag", "diag"), ("-")),
    (("diag", "diag", "col", "row"), ("-", "+", "+")),
    (("diag", "diag", "col", "row", "col"), ("-", "+", "+", "+")),
    (("diag", "diag", "col", "row", "col"), ("-", "+", "+", "-")),
    (("diag", "diag", "col", "row", "col"), ("-", "+", "-", "-")),
    (("diag", "diag", "col", "col", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            FILTER.add(p.to_hyperfield().contract(contraction_size))

print(f"Filter length={len(FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

FILTER  = UNIQ_FILTER
print(f"Filter length={len(FILTER)}")

Filter length=6268
Filter length=4273
CPU times: user 27min 12s, sys: 27.7 s, total: 27min 40s
Wall time: 27min 16s


In [33]:
EXT_FILTER = FILTER.copy()

In [34]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_FILTER = FILTER.copy()

combinations = [
    (("diag", "diag", "diag", "row", "col"), ("-", "+", "+", "+")),
    (("diag", "diag", "diag", "row", "col"), ("-", "+", "+", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_FILTER)}")


Filter length=4363
CPU times: user 13min 20s, sys: 12.6 s, total: 13min 33s
Wall time: 13min 23s


In [35]:
# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

EXT_FILTER  = UNIQ_FILTER
print(f"Filter length={len(EXT_FILTER)}")

Filter length=4320


In [36]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_EXT_FILTER = EXT_FILTER.copy()

combinations = [
    (("row", "row", "col", "diag"), ("+", "+", "+", "+")),
    (("row", "row", "col", "diag"), ("+", "+", "+", "-")),
    (("row", "row", "col", "diag"), ("+", "+", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_EXT_FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

EXT_FILTER  = UNIQ_FILTER
print(f"Filter length={len(EXT_FILTER)}")

Filter length=4396
Filter length=4369
CPU times: user 1min 52s, sys: 1.6 s, total: 1min 54s
Wall time: 1min 53s


In [37]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_EXT_FILTER = EXT_FILTER.copy()

combinations = [
    (("row", "row", "row", "row"), ("+", "+", "+", "+")),
    (("row", "row", "row", "row"), ("+", "+", "+", "-")),
    (("row", "row", "row", "row"), ("+", "+", "-", "-")),
    (("row", "row", "row", "row"), ("+", "-", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_EXT_FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

EXT_FILTER  = UNIQ_FILTER
print(f"Filter length={len(EXT_FILTER)}")

Filter length=6073
Filter length=5610
CPU times: user 2min 52s, sys: 2.53 s, total: 2min 54s
Wall time: 2min 53s


In [39]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_EXT_FILTER = EXT_FILTER.copy()

combinations = [
    (("col", "col", "col", "col"), ("+", "+", "+", "+")),
    (("col", "col", "col", "col"), ("+", "+", "+", "-")),
    (("col", "col", "col", "col"), ("+", "+", "-", "-")),
    (("col", "col", "col", "col"), ("+", "-", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_EXT_FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

EXT_FILTER  = UNIQ_FILTER
print(f"Filter length={len(EXT_FILTER)}")

Filter length=7141
Filter length=6669
CPU times: user 2min 49s, sys: 2.24 s, total: 2min 52s
Wall time: 2min 51s


In [40]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_EXT_FILTER = EXT_FILTER.copy()

combinations = [
    (("diag", "diag", "diag", "diag"), ("+", "+", "+", "+")),
    (("diag", "diag", "diag", "diag"), ("+", "+", "+", "-")),
    (("diag", "diag", "diag", "diag"), ("+", "+", "-", "-")),
    (("diag", "diag", "diag", "diag"), ("+", "-", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_EXT_FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

EXT_FILTER  = UNIQ_FILTER
print(f"Filter length={len(EXT_FILTER)}")

Filter length=6706
Filter length=6685
CPU times: user 2min 52s, sys: 2.34 s, total: 2min 55s
Wall time: 2min 54s


In [42]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_EXT_FILTER = EXT_FILTER.copy()

combinations = [
    (("col", "col", "col", "col", "col"), ("+", "+", "+", "-")),
    (("col", "col", "col", "col", "col"), ("+", "+", "-", "-")),
    (("col", "col", "col", "col", "col"), ("+", "-", "-", "-")),
    (("row", "row", "row", "row", "row"), ("+", "+", "+", "-")),
    (("row", "row", "row", "row", "row"), ("+", "+", "-", "-")),
    (("row", "row", "row", "row", "row"), ("+", "-", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_EXT_FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

EXT_FILTER  = UNIQ_FILTER
print(f"Filter length={len(EXT_FILTER)}")

Filter length=15939
Filter length=11603
CPU times: user 47min 53s, sys: 45.2 s, total: 48min 38s
Wall time: 47min 58s


In [46]:
%%time
contraction_size = 5
degree = "even"
d = contraction_size * 3 - 1
if d % 2 == 0 and degree == "odd":
    d += 5
elif d % 2 == 1 and degree == "even":
    d += 5
    
EXT_EXT_FILTER = set()

combinations = [
    (("col", "col", "col", "col", "col", "col"), ("+", "+", "+", "+", "-")),
    (("col", "col", "col", "col", "col", "col"), ("+", "+", "+", "-", "-")),
]

for comb, ops in combinations:
    for units in find_contractables(comb, ops, contraction_size):
        p = PascalForm(d, comb[0], absolute(d, units[0]))
        for base, k, op in zip(comb[1:], units[1:], ops):
            q = PascalForm(d, base, absolute(d, k))
            if op == "+":
                p += q
            elif op == "-":
                p -= q
            else:
                raise Exception("Invalid op")
        if p != LinearForm.zero(p.degree) and not p.support_pos[0] and not p.support_neg[0]:
            EXT_EXT_FILTER.add(p.to_hyperfield().contract(contraction_size))
    
print(f"Filter length={len(EXT_EXT_FILTER)}")

# here we remove redundant contracted forms
UNIQ_FILTER = set()
for p in EXT_EXT_FILTER:
    if not invert(p) in UNIQ_FILTER:
        UNIQ_FILTER.add(p)

KOKO  = UNIQ_FILTER
print(f"Filter length={len(KOKO)}")

Filter length=7649
Filter length=7331
CPU times: user 2h 51min 44s, sys: 3min 2s, total: 2h 54min 47s
Wall time: 2h 52min 13s


In [48]:
len(set(list(KOKO) + list(EXT_FILTER)))

16711

In [49]:
NEW_FILTER = KOKO.difference(EXT_FILTER)

In [20]:
%%time
VALID_SUPPORTS = []
TO_FILTER = res1
CUSTOM_FILTER = [p for p in EXT_FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

CUSTOM FILTER length is 3358
reduced 110272 supports to 5078 supports
CPU times: user 22min 32s, sys: 544 ms, total: 22min 33s
Wall time: 22min 36s


In [17]:
%%time
VALID_SUPPORTS = []
TO_FILTER = res1
CUSTOM_FILTER = [p for p in EXT_FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

CUSTOM FILTER length is 1557
reduced 110272 supports to 5435 supports
CPU times: user 13min 5s, sys: 331 ms, total: 13min 5s
Wall time: 13min 7s


In [14]:
%%time
VALID_SUPPORTS = []
TO_FILTER = res1
CUSTOM_FILTER = [p for p in EXT_FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

CUSTOM FILTER length is 537
reduced 110272 supports to 6181 supports
CPU times: user 6min 34s, sys: 825 ms, total: 6min 35s
Wall time: 6min 40s


In [44]:
%%time
import time

VALID_SUPPORTS = []
TO_FILTER = res1
CUSTOM_FILTER = [p for p in EXT_FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"{len(TO_FILTER)} supports filter")
print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

start = time.time()
for i, support_pos in enumerate(TO_FILTER):
    if i == 1000:
        end = time.time()
        elapsed = end - start
        print(f"Elapsed: {elapsed}. Estimate: {len(TO_FILTER) / 1000 * elapsed} seconds.")
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

106806 supports filter
CUSTOM FILTER length is 11603
Elapsed: 42.89458990097046. Estimate: 4581.399568963051 seconds.
reduced 106806 supports to 5779 supports
CPU times: user 1h 12min 47s, sys: 538 ms, total: 1h 12min 48s
Wall time: 1h 12min 57s


In [57]:
with open("more_contraction-master_5779supports.json", 'w') as json_file:
    json.dump(VALID_SUPPORTS, json_file, default=int)

In [50]:
GOOD_SUPPORTS = VALID_SUPPORTS.copy()

In [51]:
%%time
import time

SUPER_GOOD_SUPPORTS = []
TO_FILTER = GOOD_SUPPORTS
CUSTOM_FILTER = [p for p in NEW_FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"{len(TO_FILTER)} supports filter")
print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

start = time.time()
for i, support_pos in enumerate(TO_FILTER):
    if i == 1000:
        end = time.time()
        elapsed = end - start
        print(f"Elapsed: {elapsed}. Estimate: {len(TO_FILTER) / 1000 * elapsed} seconds.")
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        SUPER_GOOD_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(SUPER_GOOD_SUPPORTS)} supports")

5779 supports filter
CUSTOM FILTER length is 5108
Elapsed: 159.37920689582825. Estimate: 921.0524366509915 seconds.
reduced 5779 supports to 5677 supports
CPU times: user 15min 28s, sys: 276 ms, total: 15min 28s
Wall time: 15min 33s


In [56]:
with open("more_contraction-master_5677supports.json", 'w') as json_file:
    json.dump(VALID_SUPPORTS, json_file, default=int)

In [30]:
%%time

# HERE above

RES = []
TO_FILTER = VALID_SUPPORTS
CUSTOM_FILTER = [p for p in FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        RES.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(RES)} supports")

CUSTOM FILTER length is 4273
reduced 6181 supports to 4488 supports
CPU times: user 11min 4s, sys: 60.1 ms, total: 11min 4s
Wall time: 11min 5s


In [24]:
%%time
VALID_SUPPORTS = []
TO_FILTER = res2
CUSTOM_FILTER = [p for p in FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

CUSTOM FILTER length is 464
reduced 110272 supports to 6181 supports
CPU times: user 4min 11s, sys: 38.5 ms, total: 4min 11s
Wall time: 4min 11s


In [47]:
%%time
VALID_SUPPORTS = []
TO_FILTER = res1
CUSTOM_FILTER = [p for p in FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

CUSTOM FILTER length is 527
reduced 106806 supports to 7837 supports
CPU times: user 3min 5s, sys: 27.1 ms, total: 3min 5s
Wall time: 3min 6s


In [14]:
%%time
VALID_SUPPORTS = []
TO_FILTER = res2
CUSTOM_FILTER = [p for p in EXT_FILTER if not p.support_pos[0] and not p.support_neg[0]]

print(f"CUSTOM FILTER length is {len(CUSTOM_FILTER)}")

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

CUSTOM FILTER length is 489
reduced 110272 supports to 7238 supports
CPU times: user 4min 59s, sys: 44.5 ms, total: 4min 59s
Wall time: 5min


### Filter

In [75]:
%%time
res1 = countValidConfigsForContractions(6, "even", use_extra_constraints = True)
print(f"Number of configurations for even d: {len(res1)}")

res2 = countValidConfigsForContractions(6, "odd", use_extra_constraints = True)
print(f"Number of configurations for odd d: {len(res2)}")

Number of configurations for even d: 52602
Number of configurations for odd d: 52720
CPU times: user 601 ms, sys: 7.84 ms, total: 608 ms
Wall time: 605 ms


In [76]:
%%time
TO_FILTER = res1
VALID_SUPPORTS = []
CUSTOM_FILTER = [p for p in EXT_FILTER if not p.support_pos[0] and not p.support_neg[0]]

for support_pos in TO_FILTER:
    w = hyperfield_vector_from_support(contraction_size ** 2 * 3 + contraction_size * 4, support_pos, [])
    if is_root(CUSTOM_FILTER, w):
        VALID_SUPPORTS.append(support_pos)

print(f"reduced {len(TO_FILTER)} supports to {len(VALID_SUPPORTS)} supports")

reduced 52602 supports to 15253 supports
CPU times: user 7min 2s, sys: 42 ms, total: 7min 2s
Wall time: 7min 3s


**Find all supports that have size $\leq 5$ under $\chi$**

In [28]:
d0 = set([56, 57, 58, 59])
d1 = set([60, 61, 62, 63])

tmp = []

for config in VALID_SUPPORTS:
    if set(config).intersection(d0) and set(config).intersection(d1):
        tmp.append(config)
        # print(f"Config {config} has positive support 4")

len(tmp)

128

In the next cell we see that $\Lambda = \chi(\Gamma^{even} \cup \Gamma^{odd})$ contains exactly 2290 configurations.

In [29]:
def chi(config):
    t = [x for x in config]
    for x in [60, 61, 62, 63]:
        if x in t:
            t.remove(x)
            t.append(x - 4)
    t.sort()
    return tuple(set(t))    

In [30]:
# Lambda = chi(support size = 6)
Lambda = set()
for config in [c for c in VALID_SUPPORTS if c not in tmp]:
    Lambda.add(chi(config))
len(Lambda)

5756

In [31]:
c_indexes = { 48, 49, 50, 51 }
r_indexes = { 52, 53, 54, 55 }
d_indexes = { 56, 57, 58, 59 }

for x in Lambda:
    intersections = {
        "c": set(x).intersection(c_indexes),
        "r": set(x).intersection(r_indexes),
        "d": set(x).intersection(d_indexes),
    }

    for key, intersection in intersections.items():
        if len(intersection) not in { 0,1 }:
            print(f"Error: {x} and key {key} invalid")

print("Done. If no error message appeared, then Lemma 7.6 is proved. □")

Error: (34, 70, 80, 49, 51, 85) and key c invalid
Error: (73, 74, 50, 51, 25, 31) and key c invalid
Error: (8, 44, 49, 50, 25, 57) and key c invalid
Error: (32, 44, 49, 50, 21, 29) and key c invalid
Error: (32, 44, 49, 50, 51, 29) and key c invalid
Error: (34, 38, 49, 50, 51, 26) and key c invalid
Error: (65, 34, 80, 49, 51, 85) and key c invalid
Error: (6, 44, 49, 50, 57, 26) and key c invalid
Error: (16, 49, 50, 81, 85, 28) and key c invalid
Error: (8, 44, 49, 50, 56, 25) and key c invalid
Error: (50, 51, 85, 25, 90, 30) and key c invalid
Error: (76, 49, 50, 85, 26, 31) and key c invalid
Error: (49, 50, 85, 56, 29, 31) and key c invalid
Error: (9, 50, 51, 85, 90, 28) and key c invalid
Error: (34, 75, 16, 48, 49, 25) and key c invalid
Error: (34, 48, 80, 50, 56, 90) and key c invalid
Error: (34, 80, 49, 51, 85, 56) and key c invalid
Error: (8, 49, 51, 85, 55, 27) and key c invalid
Error: (34, 76, 48, 80, 50, 90) and key c invalid
Error: (32, 44, 49, 50, 56, 25) and key c invalid
Error

In [32]:
Lambda2 = []
for x in Lambda:
    if len([w for w in x if w >= 48]) > 0:
        Lambda2.append(x)
len(Lambda2)

5756

In [33]:
def find_all_configs_with_at_least_many_M(min_count_M):
    def count(supp):
        c = 0
        for x in supp:
            # only array indexes greather than 47 map to M under relcoord
            if x >= 48:
                c += 1
        return c
    return [sorted(supp) for supp in Lambda2 if count(supp) >= min_count_M]
find_all_configs_with_at_least_many_M(3)

[[34, 49, 70, 76, 80, 85],
 [16, 28, 70, 81, 85, 90],
 [34, 49, 51, 70, 80, 85],
 [25, 31, 50, 51, 73, 74],
 [29, 31, 43, 56, 75, 85],
 [9, 25, 49, 56, 65, 85],
 [29, 30, 70, 76, 85, 86],
 [8, 25, 44, 49, 50, 57],
 [8, 11, 26, 75, 85, 90],
 [33, 71, 72, 75, 80, 92],
 [21, 33, 70, 71, 80, 90],
 [15, 34, 49, 65, 71, 80],
 [29, 31, 49, 57, 75, 85],
 [11, 27, 31, 50, 73, 74],
 [8, 28, 71, 75, 85, 90],
 [9, 16, 26, 49, 65, 85],
 [33, 51, 70, 71, 80, 90],
 [21, 29, 30, 70, 73, 86],
 [8, 28, 50, 56, 85, 90],
 [29, 32, 44, 49, 50, 51],
 [7, 29, 34, 56, 68, 70],
 [21, 26, 56, 81, 85, 90],
 [31, 50, 56, 80, 85, 90],
 [29, 31, 44, 66, 67, 75],
 [9, 26, 49, 56, 76, 85],
 [8, 29, 44, 56, 68, 70],
 [8, 27, 51, 65, 85, 90],
 [10, 29, 66, 67, 81, 85],
 [9, 16, 25, 56, 85, 90],
 [21, 34, 49, 65, 80, 85],
 [33, 49, 57, 70, 71, 80],
 [26, 34, 38, 49, 50, 51],
 [29, 43, 56, 70, 73, 81],
 [27, 31, 49, 56, 75, 85],
 [6, 25, 42, 73, 74, 75],
 [8, 16, 27, 75, 85, 90],
 [25, 44, 55, 56, 81, 90],
 [16, 30, 75, 